In [103]:
import math
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn.functional as F
%matplotlib inline

In [104]:
# read in all the words
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [105]:
#build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = { s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [106]:
#build the dataset
block_size = 3 #context length:how many characters do we take to predict the next one?
def build_dataset(words):
    X,Y = [],[]
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [107]:
# ============================  now we get to the action  ==============================================

In [108]:
#utility function we will use later when comparing manual gradient to pytorch gradient
def cmp(s,dt,t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt,t.grad)  # 近似相等
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact:{str(ex):5s} | approximate:{str(app):5s} |maxdiff: {maxdiff}')
    

In [109]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(214783647)
C = torch.randn((vocab_size,n_embd),               generator = g) # 27 * 10,每个单词有一个1*10的概率
#layer 1
W1 = torch.randn((n_embd * block_size,n_hidden),   generator = g) * (5/3 * ( n_embd * block_size )**0.5)     #权重矩阵 30 * 200 ，解决tanh过度饱和的问题
b1 = torch.randn(n_hidden,                         generator = g) * 0.1
#Layer 2
W2 = torch.randn((n_hidden,vocab_size),            generator = g) * 0.1  # 解决softmax过度自信的问题
b2 = torch.randn(vocab_size,                       generator = g) * 0.1

#批量归一化中的缩放与平移参数
bngain = torch.randn((1,n_hidden)) * 0.1 +  1.0
bnbias = torch.randn((1,n_hidden)) * 0.1

# 在模型生成期间计算批量归一化参数的方法，同样也可在模型生成后统一计算批量归一化参数
bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.ones((1,n_hidden))

parameters = [C,W1,b1,W2,b2,bngain,bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True #注意不能写成require

4137


In [110]:
batch_size = 32
n = batch_size 
#minbatch construct
ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator = g)
Xb,Yb = Xtr[ix],Ytr[ix] #batch X,Y

In [130]:
# forward pass,"chunkated" into smaller steps that are possible to backward one at a time
emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0],-1) #concatenate the vectors
#Linear layer 1
hprebn = embcat @ W1 + b1
#batchNorm layer
bnmeani = 1/n*hprebn.sum(0,keepdim = True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0,keepdim=True) #notes:bessel' correction(dividing by n-1, not n)，即样本方差的估计应为总体方差的无偏估计，pytorch中实际使用无偏估计，文档中存在错误
bnvar_inv =  (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw +bnbias

#Non-linearity
h =torch.tanh(hpreact)
#linear layer 2
logits = h @ W2 + b2
#cross entropy loss
logit_maxes = logits.max(1,keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1,keepdims =True) #pytorch中为keepdim，numpy中为keepdims
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n),Yb].mean()

#pytorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs,probs,counts,counts_sum,counts_sum_inv,
         norm_logits, logit_maxes,logits,h,hpreact,bnraw,
          bnvar_inv,bnvar,bndiff2,bndiff,bnmeani,hprebn,
          embcat,emb]:
    t.retain_grad()
loss.backward()
loss



tensor(3.2912, grad_fn=<NegBackward0>)

In [131]:
# 手写的反向传播过程
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n),Yb] = -1.0 / n
dprobs = (1.0/probs) *dlogprobs
dcounts_sum_inv =  (dprobs * counts).sum(1,keepdim =True) #在梯度计算中，对于同一变量的梯度会进行累加
dcounts_sum = dcounts_sum_inv * (-1*counts_sum**-2)
dcounts = counts_sum_inv * dprobs + (torch.ones((32,27)) * dcounts_sum )
dnorm_logits = dcounts * counts  # counts = norm_logits.exp()
dlogit_maxes = -dnorm_logits.sum(1,keepdim=True)
dlogits = dnorm_logits.clone() + F.one_hot(logits.max(1).indices,num_classes = logits.shape[1])*dlogit_maxes
dh = dlogits @ W2.T # dl/dh = dl/db *W2.T  b = h@W2+b2,dl指的是损失函数的微分
dW2 = h.T @ dlogits 
db2 = dlogits.sum(0)

dhpreact = (1.0-h**2)*dh
dbngain =(dhpreact *bnraw).sum(0,keepdim=True)
dbnraw = dhpreact * bngain
dbnbias = dhpreact.sum(0,keepdim=True)
dbnvar_inv = (dbnraw * bndiff).sum(0,keepdim=True)
dbnvar = dbnvar_inv * (-0.5*((bnvar + 1e-5) ** -1.5))
dbndiff2 = dbnvar *(1/(n-1)) * torch.ones(bndiff2.shape)
dbndiff = dbnraw*bnvar_inv + dbndiff2 * 2 * bndiff
dbnmeani = -dbndiff.sum(0,keepdim=True)
dhprebn = dbndiff + dbnmeani * 1/n *torch.ones(hprebn.shape)
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros((vocab_size,n_embd))
for i in range(Xb.shape[0]):
    k = 0
    for j in Xb[i]:
        dC[j]+=demb[i][k]
        k=k+1

cmp('logprobs',dlogprobs,logprobs)
cmp('probs',dprobs,probs)
cmp('counts_sum_inv',dcounts_sum_inv,counts_sum_inv)
cmp('counts_sum',dcounts_sum,counts_sum)
cmp('counts',dcounts,counts)
cmp('norm_logits',dnorm_logits,norm_logits)
cmp('logit_maxes',dlogit_maxes,logit_maxes)
cmp('logits',dlogits,logits)
cmp('h',dh,h)
cmp('W2',dW2,W2)
cmp('b2',db2,b2)
cmp('hpreact',dhpreact,hpreact)
cmp('bngain',dbngain,bngain)
cmp('bnraw',dbnraw,bnraw)
cmp('dbnbias',dbnbias,bnbias)
cmp('bnvar_inv',dbnvar_inv,bnvar_inv)
cmp('bnvar',dbnvar,bnvar)
cmp('bndiff2',dbndiff2,bndiff2)
cmp('bndiff',dbndiff,bndiff)
cmp('bnmeani',dbnmeani,bnmeani)
cmp('hprebn',dhprebn,hprebn)
cmp('embcat',dembcat,embcat)
cmp('W1',dW1,W1)
cmp('b1',db1,b1)
cmp('emb',demb,emb)
cmp('C',dC,C)

logprobs        | exact:True  | approximate:True  |maxdiff: 0.0
probs           | exact:True  | approximate:True  |maxdiff: 0.0
counts_sum_inv  | exact:True  | approximate:True  |maxdiff: 0.0
counts_sum      | exact:True  | approximate:True  |maxdiff: 0.0
counts          | exact:True  | approximate:True  |maxdiff: 0.0
norm_logits     | exact:True  | approximate:True  |maxdiff: 0.0
logit_maxes     | exact:True  | approximate:True  |maxdiff: 0.0
logits          | exact:True  | approximate:True  |maxdiff: 0.0
h               | exact:True  | approximate:True  |maxdiff: 0.0
W2              | exact:True  | approximate:True  |maxdiff: 0.0
b2              | exact:True  | approximate:True  |maxdiff: 0.0
hpreact         | exact:True  | approximate:True  |maxdiff: 0.0
bngain          | exact:True  | approximate:True  |maxdiff: 0.0
bnraw           | exact:True  | approximate:True  |maxdiff: 0.0
dbnbias         | exact:True  | approximate:True  |maxdiff: 0.0
bnvar_inv       | exact:True  | approxim

In [132]:
#采用cross_entropy函数对进行反向传播会极大简化
#before
# logit_maxes = logits.max(1,keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1,keepdims =True) #pytorch中为keepdim，numpy中为keepdims
# counts_sum_inv = counts_sum**-1
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n),Yb].mean()

# now
loss_fast = F.cross_entropy(logits,Yb) #交叉熵损失函数
print(loss_fast.item(),'diff',(loss_fast - loss).item())

3.2911553382873535 diff 0.0


In [133]:
# 直接计算cross_entropy的反向传播梯度
dlogits = F.softmax(logits,1)
dlogits[range(n),Yb] -= 1
dlogits / n

cmp('logits',dlogits,logits)

logits          | exact:False | approximate:False |maxdiff: 0.9562046527862549


In [134]:
# 批量归一化统一函数式
#before
# bnmeani = 1/n*hprebn.sum(0,keepdim = True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0,keepdim=True) #notes:bessel' correction(dividing by n-1, not n)，即样本方差的估计应为总体方差的无偏估计，pytorch中实际使用无偏估计，文档中存在错误
# bnvar_inv =  (bnvar + 1e-5) ** -0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw +bnbias



#now:
hpreact_fast = bngain * (hprebn -  1/n*hprebn.sum(0,keepdim = True)) /torch.sqrt(hprebn.var(0,keepdim=True,unbiased = True) + 1e-5) + bnbias
print('max_diff',(hpreact_fast - hpreact).abs().max())

max_diff tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [135]:
# 批量归一化的参数统一函数式的反向传播推导
dhprebn = bngain * bnvar_inv/n * (n * dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw * (dhpreact*bnraw).sum(0))
cmp('hprebn',dhprebn,hprebn)

hprebn          | exact:False | approximate:True  |maxdiff: 2.9103830456733704e-11
